# Benchmark 2/3 — Note Detection

Component test for the offline note segmenter. Production is
`NoteDetector.detect_notes(..., model="l2")` = **ruptures PELT / L2** over the
(smoothed) pitch track; we compare it against other ruptures cost models
(`NoteBenchmarker.NOTE_METHODS`).

Per file: synthesize the etude MIDI → audio (cached), auto-tighten the range to the
MIDI, detect pitches **+ smoother**, do the app-like pitch-span prep, flag slide
frames, segment with each method, and score vs the MIDI ground truth with
`mir_eval.transcription` (onset + pitch). The score
is attached to the recording so PELT's `min_size` is derived from the shortest score
note, as in the app.

**Corpus** (`NoteBenchmarker.ETUDE_DATASETS`): monophonic violin etudes Kayser + Wohlfahrt
(Paganini excluded — double-stops).

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import sys
sys.path.insert(0, "../benchmarks")
from NoteBenchmarker import NoteBenchmarker
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
bm = NoteBenchmarker(max_tracks=8, onset_tolerance=0.05)

## Method comparison — PELT L2 vs alternatives
Mean Precision / Recall / F-measure over the corpus, per method (**pelt-l2** = production).

In [ ]:
df = pd.concat([bm.bench_note_dataset(ds, write=True) for ds in bm.ETUDE_DATASETS], ignore_index=True)
df.groupby("method")[["Precision","Recall","F-measure","Average Overlap Ratio"]].mean().sort_values("F-measure", ascending=False)

Per-dataset F-measure by method:

In [ ]:
df.pivot_table(index="method", columns="dataset", values="F-measure", aggfunc="mean")

## Onset-tolerance sweep
How the ranking holds up as the matching tolerance loosens (50 ms = mir_eval standard).

In [ ]:
sweep = {}
for onset_tolerance in (0.025, 0.05, 0.10):
    d = pd.concat([bm.bench_note_dataset(ds, max_tracks=3, onset_tolerance=onset_tolerance, verbose=False)
                   for ds in bm.ETUDE_DATASETS], ignore_index=True)
    sweep[onset_tolerance] = d.groupby("method")["F-measure"].mean()
pd.DataFrame(sweep).sort_values(0.05, ascending=False)

## Production timing
The note benchmark uses cached production pitch data with smoothing; this summarizes the note-detection stage timing separately.

In [ ]:
timing_df = pd.concat([bm.bench_note_dataset(ds, max_tracks=4, verbose=False)
                       for ds in bm.ETUDE_DATASETS], ignore_index=True)
timing_df[timing_df.method=="pelt-l2"][["Precision","Recall","F-measure","note_compute_time"]].mean()